# PDF READER TESTING

In [2]:
import config
import os
import sys
import logging
import random
from torch.cuda import empty_cache, manual_seed

import asyncio
import nest_asyncio

# Logging
logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger().addHandler(logging.StreamHandler(stream=sys.stdout))

# API Keys
os.environ['OPENAI_API_KEY'] = config.openai_api_key
os.environ['GROQ_API_KEY'] = config.groq_api_key

# CUDA GPU memory avoid fragmentation.
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'  # CUDA GPU memory avoid fragmentation.
os.environ['MAX_SPLIT_SIZE_MB'] = '128'
os.environ['SCARF_NO_ANALYTICS'] = 'true'  # get rid of data collection from Unstructured-IO. be sure this is set, otherwise unstructured will take ages pinging the telemetry.
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

# Asyncio: fix some issues with nesting https://github.com/run-llama/llama_index/issues/9978
nest_asyncio.apply()

# Set seeds
if (random.getstate() is None or random.getstate() != 31415926):
    random.seed(31415926)
    manual_seed(31415926)

In [3]:

from streamlit_pdf_viewer import pdf_viewer
from streamlit import session_state as ss
import streamlit as st
from io import BytesIO  # type hinting for streamlit documents

from typing import Optional, Any, Dict, List, Set, Tuple, Callable, Dict, cast

import pandas as pd

from joblib import Parallel, delayed

from llama_index.core import Settings
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.core import VectorStoreIndex, get_response_synthesizer
from llama_index.core.readers.base import BaseReader
from llama_index.core.schema import QueryBundle
from llama_index.core.response.notebook_utils import display_source_node
from llama_index.core.postprocessor import SimilarityPostprocessor, SentenceEmbeddingOptimizer
from llama_index.core.response_synthesizers import ResponseMode

# These are mine, not llama-index's
from pdf_reader import UnstructuredPDFReader
from pdf_reader_utils import clean_abbreviations, dedupe_title_chunks, combine_listitem_chunks, remove_header_footer_repeated, chunk_by_header, UnstructuredPDFPostProcessor, RegexMetadataAdder, KeywordMetadataAdder, TextSummaryMetadataAdder, TableSummaryMetadataAdder, ImageSummaryMetadataAdder
from pdf_reader_utils import DATE_REGEX, TIME_REGEX, EMAIL_REGEX, MAIL_ADDR_REGEX, PHONE_REGEX
from storage import get_vector_store, pdf_to_storage
from parsers import get_parser
from retriever import get_retriever
from prompts import get_qa_prompt, get_refine_prompt
from models import get_sat_sentence_splitter, get_embedder, get_reranker, get_llm, get_multimodal_llm
from engine import get_engine
from obs_logging import get_callback_manager, get_obs

ImportError: cannot import name 'UnstructuredPDFPostProcessor' from 'pdf_reader_utils' (c:\Users\jdwh0\DS\AutoDocLifter\pdf_reader_utils.py)

In [ ]:
vision_llm = get_multimodal_llm(
    # model_name='dwb2023/phi-3-vision-128k-instruct-quantized',
    # tokenizer_model_name='microsoft/Phi-3-vision-128k-instruct',
)

2024-07-21 17:53:53.363 
  command:

    streamlit run c:\Users\jdwh0\mambaforge\envs\autodoc\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]


In [ ]:
llm = get_llm()
Settings.llm = llm

In [ ]:
sent_splitter = get_sat_sentence_splitter('sat-3l-sm')

In [ ]:
embed_model = get_embedder()
Settings.embed_model = embed_model

INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: mixedbread-ai/mxbai-embed-large-v1
Load pretrained SentenceTransformer: mixedbread-ai/mxbai-embed-large-v1


c:\Users\jdwh0\mambaforge\envs\autodoc\Lib\site-packages\torch\nn\modules\module.py:1159: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\cb\pytorch_1000000000000\work\c10/cuda/CUDAAllocatorConfig.h:28.)
  return t.to(


INFO:sentence_transformers.SentenceTransformer:2 prompts are loaded, with the keys: ['query', 'text']
2 prompts are loaded, with the keys: ['query', 'text']


In [ ]:
callback_manager = get_callback_manager()
Settings.callback_manager = callback_manager

In [ ]:
node_parser = get_parser(embed_model=Settings.embed_model, sentence_model=sent_splitter, callback_manager=Settings.callback_manager)
Settings.node_parser = node_parser

In [ ]:
obs = get_obs()

In [8]:
pdf_reader = UnstructuredPDFReader()

## Read in PDF

In [ ]:
import gc
import torch
gc.collect()
torch.cuda.empty_cache()

In [4]:
# from unstructured.partition.pdf import partition_pdf
# pdf_elements = partition_pdf(
#     filename='data/The-Yellow-Wall-Paper.pdf',
#     # file=pdf_file,
#     unique_element_ids=True,  # UUIDs that are unique for each element
#     strategy="hi_res",  # auto: it decides, hi_res: detectron2, but issues with multi-column, ocr_only: pytesseract, fast: pdfminer
#     hi_res_model_name='yolox',
#     include_page_breaks=False,
#     # metadata_filename=pdf_file_path,
#     infer_table_structure=True,
#     extract_images_in_pdf=True,
#     extract_image_block_types=['Image', 'Table', 'Formula'],  # element types to save as images
#     extract_image_block_to_payload=False,  # needs to be false; we'll convert into base64 later.
#     extract_forms=False,  # not currently available
#     # extract_image_block_output_dir=os.path.join(os.path.dirname(os.path.abspath(__file__)), 'data/pdfimgs/')
# )

In [5]:
# for element in pdf_elements:
#     print("----------------------------")
#     print(element.category)
#     print(element.text)
#     print(element.metadata.to_dict())

In [6]:
# pdf_elements[0].metadata.to_dict()

In [10]:
pdf_chunks = pdf_reader.load_data(file_path='data/Nash_NonCooperative_Games.pdf')#'data/input.pdf')

INFO:pikepdf._core:pikepdf C++ to Python logger bridge initialized
pikepdf C++ to Python logger bridge initialized
INFO:unstructured_inference:Reading PDF for file: data/Nash_NonCooperative_Games.pdf ...
Reading PDF for file: data/Nash_NonCooperative_Games.pdf ...
INFO:unstructured_inference:Loading the Table agent ...
Loading the Table agent ...
INFO:unstructured_inference:Loading the table structure model ...
Loading the table structure model ...
INFO:timm.models._builder:Loading pretrained weights from Hugging Face hub (timm/resnet18.a1_in1k)
Loading pretrained weights from Hugging Face hub (timm/resnet18.a1_in1k)
INFO:timm.models._hub:[timm/resnet18.a1_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
[timm/resnet18.a1_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
INFO:timm.models._builder:Missing keys (fc.weight, fc.bias) discovered while load

Some weights of the model checkpoint at microsoft/table-transformer-structure-recognition were not used when initializing TableTransformerForObjectDetection: ['model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked']
- This IS expected if you are initializing TableTransformerForObjectDetection from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TableTransformerForObjectDetection from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [11]:
for node in pdf_chunks:
    print("----------------------------")
    # print([subnode.metadata['type'] for subnode in node.metadata['orig_nodes']])
    print (node.metadata['type'])
    print(node.text)
    print(node.metadata)

----------------------------
Title
Non-Cooperative Games
{'type': 'Title', 'filename': 'Nash_NonCooperative_Games.pdf', 'coordinates': {'points': ((198.17764282226562, 337.8756103515625), (198.17764282226562, 367.3468322753906), (555.32177734375, 367.3468322753906), (555.32177734375, 337.8756103515625)), 'system': 'PixelSpace', 'layout_width': 1700, 'layout_height': 2200}, 'page_number': [1], 'detection_class_prob': 0.8252522945404053, 'languages': ['eng']}
----------------------------
Image

{'type': 'Image', 'filename': 'Nash_NonCooperative_Games.pdf', 'coordinates': {'points': ((1318.124267578125, 171.89356994628906), (1318.124267578125, 401.4829406738281), (1507.5435791015625, 401.4829406738281), (1507.5435791015625, 171.89356994628906)), 'system': 'PixelSpace', 'layout_width': 1700, 'layout_height': 2200}, 'page_number': [1], 'detection_class_prob': 0.8033626675605774, 'languages': ['eng']}
----------------------------
NarrativeText
John Nash
{'type': 'NarrativeText', 'filename': 

In [12]:
pdf_chunks = clean_abbreviations(pdf_chunks)
pdf_chunks = dedupe_title_chunks(pdf_chunks)
pdf_chunks = combine_listitem_chunks(pdf_chunks)

In [13]:
pdf_chunks = remove_header_footer_repeated(pdf_chunks, fuzz_threshold=80, window_size=3)

In [14]:
for node in pdf_chunks:
    print("----------------------------")
    # print([subnode.metadata['type'] for subnode in node.metadata['orig_nodes']])
    print (node.metadata['type'])
    print(node.text)

----------------------------
Title
Non-Cooperative Games
----------------------------
Image

----------------------------
NarrativeText
John Nash
----------------------------
NarrativeText
The Annals of Mathematics, Second Series, Volume 54, Issue 2 (Sep., 1951), 286-295.
----------------------------
NarrativeText
Stable URL:
----------------------------
NarrativeText
http:/Ainks.jstor.org/sici?sici=0003-486X %28 195 109%292%3A54%3A2%3C286%3ANG%3E2.0.CO%3B2-G
----------------------------
NarrativeText
Your use of the JSTOR archive indicates your acceptance of JSTOR’s Terms and Conditions of Use, available at http://www .jstor.org/about/terms.html. JSTOR’s Terms and Conditions of Use provides, in part, that unless you have obtained prior permission, you may not download an entire issue of a journal or multiple copies of articles, and you may use content in the JSTOR archive only for your personal, non-commercial use.
----------------------------
NarrativeText
Each copy of any part of a 

In [16]:
# keyword_adder = KeywordMetadataAdder()
# pdf_chunks = keyword_adder(pdf_chunks)

In [17]:
# table_adder = TableSummaryMetadataAdder()
# pdf_chunks = table_adder(pdf_chunks)

In [18]:
# image_adder = ImageSummaryMetadataAdder(llm=vision_llm)
# pdf_chunks = image_adder(pdf_chunks)

In [20]:
for node in pdf_chunks:
    if ('parent_id' not in node.metadata):
        print("----------------------------")
        print([subnode.metadata['type'] for subnode in node.metadata.get('orig_nodes', [])])
        print(node.metadata)
        # print (node.metadata)
        print(node.text)

----------------------------
[]
{'type': 'Title', 'filename': 'Nash_NonCooperative_Games.pdf', 'coordinates': {'points': ((198.17764282226562, 337.8756103515625), (198.17764282226562, 367.3468322753906), (555.32177734375, 367.3468322753906), (555.32177734375, 337.8756103515625)), 'system': 'PixelSpace', 'layout_width': 1700, 'layout_height': 2200}, 'page_number': [1], 'detection_class_prob': 0.8252522945404053, 'languages': ['eng']}
Non-Cooperative Games
----------------------------
[]
{'type': 'Image', 'filename': 'Nash_NonCooperative_Games.pdf', 'coordinates': {'points': ((1318.124267578125, 171.89356994628906), (1318.124267578125, 401.4829406738281), (1507.5435791015625, 401.4829406738281), (1507.5435791015625, 171.89356994628906)), 'system': 'PixelSpace', 'layout_width': 1700, 'layout_height': 2200}, 'page_number': [1], 'detection_class_prob': 0.8033626675605774, 'languages': ['eng']}

----------------------------
[]
{'type': 'NarrativeText', 'filename': 'Nash_NonCooperative_Games.

In [21]:
pdf_chunks_1 = chunk_by_header(pdf_chunks, 1000, True)

AttributeError: 'dict' object has no attribute 'parent_id'

In [30]:
for node in pdf_chunks_1:
    print("----------------------------")
    # print([subnode.metadata['type'] for subnode in node.metadata['orig_nodes']])
    print (node.metadata['type'])
    print(node.text)

----------------------------
Image
© Levy Economics © Institute — of Bard College _
----------------------------
Composite
Working Paper Number74 The Financial Instability Hypothesis*
by Hyman P. Minsky The Jerome Levy Economics Institute of Bard College
 May 1992 * Prepared for Handbook of Radical Political Economy, edited by Philip Arestis and Malcolm Sawyer, Edward Elgar: Aldershot, 1993.
The Levy Economics Institute Working Paper Collection presents research in progress by Levy Institute scholars and conference participants. The purpose of the series is to disseminate ideas to and elicit comments from academics and professionals.
Levy Economics Institute of Bard College, founded in 1986, is a nonprofit, nonpartisan, independently funded research organization devoted to public service. Through scholarship and economic research it generates viable, effective public policy responses to important economic problems that profoundly affect the quality of life in the United States and abro

In [31]:
pdf_chunks_2 = node_parser.get_nodes_from_documents(pdf_chunks_1)

INFO:obs_logging:-----------------------
-----------------------
INFO:obs_logging:e7fab21d-5928-4ca4-a29f-7b35e8d8266a
e7fab21d-5928-4ca4-a29f-7b35e8d8266a
INFO:obs_logging:2024-07-21 18:22:33.491598
2024-07-21 18:22:33.491598
INFO:obs_logging:BaseEmbedding.get_text_embedding_batch-009cf286-bc52-420f-b04b-c6b94b694c48
BaseEmbedding.get_text_embedding_batch-009cf286-bc52-420f-b04b-c6b94b694c48
INFO:obs_logging:Event type: EmbeddingStartEvent
Event type: EmbeddingStartEvent
INFO:obs_logging:{'model_name': 'mixedbread-ai/mxbai-embed-large-v1', 'embed_batch_size': 10, 'num_workers': None, 'max_length': 512, 'normalize': True, 'query_instruction': None, 'text_instruction': None, 'cache_folder': None, 'class_name': 'HuggingFaceEmbedding'}
{'model_name': 'mixedbread-ai/mxbai-embed-large-v1', 'embed_batch_size': 10, 'num_workers': None, 'max_length': 512, 'normalize': True, 'query_instruction': None, 'text_instruction': None, 'cache_folder': None, 'class_name': 'HuggingFaceEmbedding'}
INFO:obs

-----------------------


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:obs_logging:-----------------------
-----------------------
INFO:obs_logging:cc2dbae3-d592-43cd-90c6-e97c62a588bb
cc2dbae3-d592-43cd-90c6-e97c62a588bb
INFO:obs_logging:2024-07-21 18:22:36.498213
2024-07-21 18:22:36.498213
INFO:obs_logging:BaseEmbedding.get_text_embedding_batch-009cf286-bc52-420f-b04b-c6b94b694c48
BaseEmbedding.get_text_embedding_batch-009cf286-bc52-420f-b04b-c6b94b694c48
INFO:obs_logging:Event type: EmbeddingEndEvent
Event type: EmbeddingEndEvent
INFO:obs_logging:['© Levy Economics © Institute — of Bard College _']
['© Levy Economics © Institute — of Bard College _']
INFO:obs_logging:[-0.019113196060061455, 0.002611134434118867, 0.03721960633993149, -0.04234164580702782, 0.003106417367234826]
[-0.019113196060061455, 0.002611134434118867, 0.03721960633993149, -0.04234164580702782, 0.003106417367234826]
INFO:obs_logging:-----------------------
-----------------------
INFO:obs_logging:-----------------------
-----------------------
INFO:obs_logging:05a14d3b-6d2d-43c1

c:\Users\jdwh0\mambaforge\envs\autodoc\Lib\site-packages\transformers\models\bert\modeling_bert.py:439: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


...

INFO:obs_logging:-----------------------
-----------------------
INFO:obs_logging:1c813e6a-60ea-46fb-a65b-82746188f869
1c813e6a-60ea-46fb-a65b-82746188f869
INFO:obs_logging:2024-07-21 18:22:40.767048
2024-07-21 18:22:40.767048
INFO:obs_logging:BaseEmbedding.get_text_embedding_batch-21bf8bcf-4d8b-4ea2-83bd-c63cb8501fcd
BaseEmbedding.get_text_embedding_batch-21bf8bcf-4d8b-4ea2-83bd-c63cb8501fcd
INFO:obs_logging:Event type: EmbeddingEndEvent
Event type: EmbeddingEndEvent
INFO:obs_logging:['Keynes, John Maynard. 1972. Writings of John Maynard Keynes, Volume IX. Martins Press, for the Royal Economic Society, London and Basingstoke, p 151 Kindleberger, Charles 1978. Manias, Panics and Crashes. New York, Basic Books Levy S. Jay and David A. 1983. ', '1972. Writings of John Maynard Keynes, Volume IX. Martins Press, for the Royal Economic Society, London and Basingstoke, p 151 Kindleberger, Charles 1978. Manias, Panics and Crashes. New York, Basic Books Levy S. Jay and David A. 1983. American S

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:obs_logging:-----------------------
-----------------------
INFO:obs_logging:998fa041-0127-49c3-93ac-ceec01b91b77
998fa041-0127-49c3-93ac-ceec01b91b77
INFO:obs_logging:2024-07-21 18:22:40.885193
2024-07-21 18:22:40.885193
INFO:obs_logging:BaseEmbedding.get_text_embedding_batch-21bf8bcf-4d8b-4ea2-83bd-c63cb8501fcd
BaseEmbedding.get_text_embedding_batch-21bf8bcf-4d8b-4ea2-83bd-c63cb8501fcd
INFO:obs_logging:Event type: EmbeddingEndEvent
Event type: EmbeddingEndEvent
INFO:obs_logging:['John Maynard Keynes. Columbia University Press.\nMinsky, Hyman P. 1986. University Press. Stabilizing An Unstable Economy. Yale Schumpeter, Joseph A. ', 'Columbia University Press.\nMinsky, Hyman P. 1986. University Press. Stabilizing An Unstable Economy. Yale Schumpeter, Joseph A. 1934. ', 'Minsky, Hyman P. 1986. University Press. Stabilizing An Unstable Economy. Yale Schumpeter, Joseph A. 1934. Theory of Economic Development. ', '1986. University Press. Stabilizing An Unstable Economy. Yale Schumpeter

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:obs_logging:-----------------------
-----------------------
INFO:obs_logging:7b6fa837-564e-4f81-900f-ec72f3462f9d
7b6fa837-564e-4f81-900f-ec72f3462f9d
INFO:obs_logging:2024-07-21 18:22:40.996652
2024-07-21 18:22:40.996652
INFO:obs_logging:BaseEmbedding.get_text_embedding_batch-21bf8bcf-4d8b-4ea2-83bd-c63cb8501fcd
BaseEmbedding.get_text_embedding_batch-21bf8bcf-4d8b-4ea2-83bd-c63cb8501fcd
INFO:obs_logging:Event type: EmbeddingEndEvent
Event type: EmbeddingEndEvent
INFO:obs_logging:['Harvard University Press Wolfson, Martin H. 1986. Financial Crises. Armonk New York, M.E. Sharpe Inc.', '1986. Financial Crises. Armonk New York, M.E. Sharpe Inc.']
['Harvard University Press Wolfson, Martin H. 1986. Financial Crises. Armonk New York, M.E. Sharpe Inc.', '1986. Financial Crises. Armonk New York, M.E. Sharpe Inc.']
INFO:obs_logging:[-0.032942552119493484, -0.0123163852840662, 0.06232699006795883, -0.02150612138211727, 0.034145087003707886]
[-0.032942552119493484, -0.0123163852840662, 0.06

In [33]:
for node in pdf_chunks_2:
    print("----------------------------")
    # print([subnode.metadata['type'] for subnode in node.metadata['orig_nodes']])
    print (node.metadata['type'])
    print(node.text)
    print(len(node.text))

----------------------------
Image
© Levy Economics © Institute — of Bard College _
48
----------------------------
Composite
Working Paper Number74 The Financial Instability Hypothesis*
by Hyman P. Minsky The Jerome Levy Economics Institute of Bard College
 May 1992 * Prepared for Handbook of Radical Political Economy, edited by Philip Arestis and Malcolm Sawyer, Edward Elgar: Aldershot, 1993.
The Levy Economics Institute Working Paper Collection presents research in progress by Levy Institute scholars and conference participants. The purpose of the series is to disseminate ideas to and elicit comments from academics and professionals.
Levy Economics Institute of Bard College, founded in 1986, is a nonprofit, nonpartisan, independently funded research organization devoted to public service. Through scholarship and economic research it generates viable, effective public policy responses to important economic problems that profoundly affect the quality of life in the United States and a

In [ ]:
pdf_chunks

[TextNode(id_='45cd0591-b001-498d-9795-cdc8e2cfe52d', embedding=None, metadata={'type': 'FigureCaption', 'filename': 'The-Yellow-Wall-Paper.pdf', 'coordinates': {'points': ((384.9053649902344, 1102.0520047222221), (384.9053649902344, 1120.773681640625), (845.736083984375, 1120.773681640625), (845.736083984375, 1102.0520047222221)), 'system': 'PixelSpace', 'layout_width': 1238, 'layout_height': 1890}, 'page_number': [1], 'detection_class_prob': 0.7671131491661072, 'languages': ['eng'], 'keyword_metadata': 'atrocious nursery, window, sitting, nursery, atrocious'}, excluded_embed_metadata_keys=['type', 'parent_id', 'depth', 'filename', 'coordinates', 'page number', 'original_text', 'window', 'link_texts', 'link_urls', 'link_start_indexes', 'orig_nodes', 'orignal_table_text', 'languages', 'detection_class_prob'], excluded_llm_metadata_keys=['type', 'parent_id', 'depth', 'filename', 'coordinates', 'link_texts', 'link_urls', 'link_start_indexes', 'orig_nodes', 'orignal_table_text', 'language